# Шаг 1. Понимаем FAR-Trans

Сегодняшняя цель: прочитать таблицу, понять одну запись и сформулировать, что будет предсказывать модель.

Источник: [Университет Глазго](https://researchdata.gla.ac.uk/1658/).
Публикация: [Sanz-Cruzado, Droukas, McCreadie, 2024](https://arxiv.org/abs/2407.08692).

Используем `pandas` — библиотеку для чтения и обработки таблиц. Код ниже уже выполнен; его можно запускать повторно. Путь в следующей ячейке относится к этому компьютеру: при переносе ноутбука его нужно заменить.


In [1]:
import pandas as pd

data_dir = '/Users/annazhivikhina/Documents/Codex/2026-10-11/t-lab-recsys-ml-train-validation/work/data/FAR-Trans'

transactions = pd.read_csv(
    data_dir + "/transactions.csv",
    dtype={"customerID": "string", "ISIN": "string"},
)

print(transactions.head(3).to_string(index=False))


          customerID         ISIN  transactionID transactionType  timestamp  totalValue  units          channel marketID
00017496858921195E5A GRS434003000        7590224             Buy 2020-03-27     11000.0 5000.0 Internet Banking     XATH
00017496858921195E5A GRS434003000        7607029            Sell 2020-04-06     12080.0 5000.0 Internet Banking     XATH
00017496858921195E5A GRS434003000        7634872             Buy 2020-04-24     13400.0 5000.0 Internet Banking     XATH


`transactions` — переменная, содержащая таблицу.

- `customerID`: обезличенный идентификатор клиента. Храним как текст, чтобы сохранить начальные нули.
- `ISIN`: код финансового актива.
- `transactionType`: покупка (`Buy`) или продажа (`Sell`).
- `timestamp`: дата сделки; точного времени внутри дня здесь нет.
- `totalValue`: сумма сделки в евро в обработанном авторами наборе.
- `units`: количество единиц актива.
- `channel`: канал совершения сделки.

`head(3)` показывает первые три строки, а `to_string(index=False)` выводит их без дополнительного номера строки.

Одна строка описывает состоявшуюся сделку. Это ещё не обучающий пример «клиент — возможный актив — момент прогноза».


In [2]:
client_id = "00017496858921195E5A"

client_history = transactions[
    transactions["customerID"] == client_id
]

print(client_history[["timestamp", "transactionType", "ISIN"]])

        timestamp transactionType          ISIN
0      2020-03-27             Buy  GRS434003000
1      2020-04-06            Sell  GRS434003000
2      2020-04-24             Buy  GRS434003000
3      2020-05-07            Sell  GRS434003000
4      2020-05-15             Buy  GRS434003000
...           ...             ...           ...
34101  2022-09-30             Buy  GRS473003002
34102  2022-10-03             Buy  GRS473003002
34103  2022-10-06             Buy  GRS473003002
34104  2022-11-23            Sell  GRS473003002
34105  2022-11-24            Sell  GRS473003002

[124 rows x 3 columns]


In [3]:
# История за март: доступна для прогноза на 1 апреля.
client_dates = pd.to_datetime(client_history["timestamp"])

march = client_history[
    (client_dates >= "2020-03-01")
    & (client_dates < "2020-04-01")
]

print(march[["timestamp", "transactionType", "ISIN"]])


    timestamp transactionType          ISIN
0  2020-03-27             Buy  GRS434003000


In [4]:
client_dates = pd.to_datetime(client_history["timestamp"])

april = client_history[
    (client_dates >= "2020-04-01")
    & (client_dates < "2020-05-01")
]

print(april[["timestamp", "transactionType", "ISIN"]])

    timestamp transactionType          ISIN
1  2020-04-06            Sell  GRS434003000
2  2020-04-24             Buy  GRS434003000


In [5]:
transactions["timestamp"] = pd.to_datetime(transactions["timestamp"], errors="raise")

print("Строк:", len(transactions))
print("Клиентов:", transactions["customerID"].nunique())
print("Активов со сделками:", transactions["ISIN"].nunique())
print("Первая дата:", transactions["timestamp"].min().date())
print("Последняя дата:", transactions["timestamp"].max().date())
print("\nВиды сделок:")
print(transactions["transactionType"].value_counts().to_string())


Строк: 388048
Клиентов: 29090
Активов со сделками: 320
Первая дата: 2018-01-02
Последняя дата: 2022-11-30

Виды сделок:
transactionType
Buy     228913
Sell    159135


In [6]:
march_buys = march[
    march["transactionType"] == "Buy"
]

march_buy_count = len(march_buys)

print("Покупок за март:", march_buy_count)

Покупок за март: 1


`pd.to_datetime` превращает текстовые даты в даты Python/pandas. `nunique()` считает разные значения, `value_counts()` — сколько раз встречается каждое значение.

Точные числа получены из скачанного файла. Они немного отличаются от таблицы в статье (388 049 записей и 321 актив); версия файлов и их проверка важны для воспроизводимости.


In [7]:
april_buys = april[
    april["transactionType"] == "Buy"
]

april_buy_count = len(april_buys)

print("Покупок за апрель:", april_buy_count)

Покупок за апрель: 1


In [8]:
asset_id = "GRS434003000"

target_buys = april_buys[
    april_buys["ISIN"] == asset_id
]

if len(target_buys) > 0:
    label = 1
else:
    label = 0

print("Правильный ответ:", label)

Правильный ответ: 1


In [9]:
example = pd.DataFrame([{
    "customerID": client_id,
    "ISIN": asset_id,
    "prediction_date": "2020-04-01",
    "user_buy_count_1m": len(march_buys),
    "label": label
}])

print(example.to_string(index=False))

          customerID         ISIN prediction_date  user_buy_count_1m  label
00017496858921195E5A GRS434003000      2020-04-01                  1      1


In [10]:
other_asset_id = "GRF000014008"

other_target_buys = april_buys[
    april_buys["ISIN"] == other_asset_id
]

print(other_target_buys)

if len(other_target_buys) > 0:
    other_label = 1
else:
    other_label = 0

print("Ответ для второго актива:", other_label)

Empty DataFrame
Columns: [customerID, ISIN, transactionID, transactionType, timestamp, totalValue, units, channel, marketID]
Index: []
Ответ для второго актива: 0


In [11]:
other_example = example.copy()
other_example["ISIN"] = other_asset_id
other_example["label"] = other_label

examples = pd.concat(
    [example, other_example],
    ignore_index=True
)

print(examples.to_string(index=False))

          customerID         ISIN prediction_date  user_buy_count_1m  label
00017496858921195E5A GRS434003000      2020-04-01                  1      1
00017496858921195E5A GRF000014008      2020-04-01                  1      0


In [12]:
asset_buy_counts = []

for isin in examples["ISIN"]:
    past_asset_buys = march_buys[
        march_buys["ISIN"] == isin
    ]
    asset_buy_counts.append(len(past_asset_buys))

examples["user_item_buy_count_1m"] = asset_buy_counts

print(
    examples[
        ["ISIN", "user_item_buy_count_1m", "label"]
    ].to_string(index=False)
)

        ISIN  user_item_buy_count_1m  label
GRS434003000                       1      1
GRF000014008                       0      0


In [17]:
transactions["timestamp"] = pd.to_datetime(transactions["timestamp"])

def get_past_buys(client_id, prediction_date):
    cutoff = pd.Timestamp(prediction_date)
    start = cutoff - pd.DateOffset(months=1)

    past_buys = transactions[
        (transactions["customerID"] == client_id)
        & (transactions["transactionType"] == "Buy")
        & (transactions["timestamp"] >= start)
        & (transactions["timestamp"] < cutoff)
    ]

    return past_buys

In [19]:
history = get_past_buys(
    client_id,
    "2020-05-01"
)

print(history[["timestamp", "transactionType", "ISIN"]])

   timestamp transactionType          ISIN
2 2020-04-24             Buy  GRS434003000


In [20]:
def get_label(client_id, asset_id, prediction_date):
    cutoff = pd.Timestamp(prediction_date)
    end = cutoff + pd.DateOffset(months=1)

    future_buys = transactions[
        (transactions["customerID"] == client_id)
        & (transactions["ISIN"] == asset_id)
        & (transactions["transactionType"] == "Buy")
        & (transactions["timestamp"] >= cutoff)
        & (transactions["timestamp"] < end)
    ]

    if len(future_buys) > 0:
        return 1
    else:
      return 0

In [21]:
print(get_label(client_id, asset_id, "2020-05-01"))
print(get_label(client_id, other_asset_id, "2020-05-01"))

1
0


In [22]:
def make_example(client_id, asset_id, prediction_date):
    past_buys = get_past_buys(client_id, prediction_date)

    past_asset_buys = past_buys[
        past_buys["ISIN"] == asset_id
    ]

    return {
        "customerID": client_id,
        "ISIN": asset_id,
        "prediction_date": prediction_date,
        "user_buy_count_1m": len(past_buys),
        "user_item_buy_count_1m": len(past_asset_buys),
        "label": get_label(client_id, asset_id, prediction_date)
    }

In [23]:
auto_examples = pd.DataFrame([
    make_example(client_id, asset_id, "2020-05-01"),
    make_example(client_id, other_asset_id, "2020-05-01")
])

print(auto_examples.to_string(index=False))

          customerID         ISIN prediction_date  user_buy_count_1m  user_item_buy_count_1m  label
00017496858921195E5A GRS434003000      2020-05-01                  1                       1      1
00017496858921195E5A GRF000014008      2020-05-01                  1                       0      0


In [24]:
known_history = transactions[
    transactions["timestamp"] < "2020-04-01"
]

client_ids = (
    known_history["customerID"]
    .drop_duplicates()
    .head(3)
    .tolist()
)

asset_ids = [asset_id, other_asset_id]

prediction_dates = [
    "2020-04-01",
    "2020-05-01",
    "2020-06-01"
]

In [25]:
rows = []

for date in prediction_dates:
    for client in client_ids:
        for asset in asset_ids:
            rows.append(
                make_example(client, asset, date)
            )

mini_dataset = pd.DataFrame(rows)

print(mini_dataset.to_string(index=False))
print("Количество строк:", len(mini_dataset))

          customerID         ISIN prediction_date  user_buy_count_1m  user_item_buy_count_1m  label
00017496858921195E5A GRS434003000      2020-04-01                  1                       1      1
00017496858921195E5A GRF000014008      2020-04-01                  1                       0      0
0004718496C71D464F57 GRS434003000      2020-04-01                  0                       0      0
0004718496C71D464F57 GRF000014008      2020-04-01                  0                       0      0
001270E70A4ECA1CD664 GRS434003000      2020-04-01                  2                       1      0
001270E70A4ECA1CD664 GRF000014008      2020-04-01                  2                       0      0
00017496858921195E5A GRS434003000      2020-05-01                  1                       1      1
00017496858921195E5A GRF000014008      2020-05-01                  1                       0      0
0004718496C71D464F57 GRS434003000      2020-05-01                  0                       0      0


In [ ]:
train = mini_dataset[
    mini_dataset["prediction_date"] == "2020-04-01"
].copy()

validation = mini_dataset[
    mini_dataset["prediction_date"] == "2020-05-01"
].copy()

test = mini_dataset[
    mini_dataset["prediction_date"] == "2020-06-01"
].copy()

print("Обучение:", len(train))
print("Выбор решения:", len(validation))
print("Итоговая проверка:", len(test))

Обучение: 6
Выбор решения: 6
Итоговая проверка: 6


In [27]:
# Покупки за полгода до первого прогноза
selection_buys = transactions[
    (transactions["transactionType"] == "Buy")
    & (transactions["timestamp"] >= "2019-10-01")
    & (transactions["timestamp"] < "2020-04-01")
]

# Случайно выбираем 500 разных клиентов
large_client_ids = (
    selection_buys["customerID"]
    .drop_duplicates()
    .sample(n=500, random_state=42)
    .tolist()
)

# Выбираем 50 активов с наибольшим числом покупок
large_asset_ids = (
    selection_buys["ISIN"]
    .value_counts()
    .head(50)
    .index.tolist()
)

print("Клиентов:", len(large_client_ids))
print("Активов:", len(large_asset_ids))

Клиентов: 500
Активов: 50


In [28]:
# Все покупки выбранных клиентов
large_buys = transactions[
    (transactions["transactionType"] == "Buy")
    & transactions["customerID"].isin(large_client_ids)
].copy()

# Только март: прошлое для прогноза на 1 апреля
march_large_buys = large_buys[
    (large_buys["timestamp"] >= "2020-03-01")
    & (large_buys["timestamp"] < "2020-04-01")
]

# Сколько всего покупок у каждого клиента
user_counts = (
    march_large_buys.groupby("customerID")
    .size()
    .reset_index(name="user_buy_count_1m")
)

# Сколько покупок конкретного актива у каждого клиента
pair_counts = (
    march_large_buys.groupby(["customerID", "ISIN"])
    .size()
    .reset_index(name="user_item_buy_count_1m")
)

print(user_counts.head())
print(pair_counts.head())

             customerID  user_buy_count_1m
0  00523D5E7518C6165B32                  2
1  0203622547FA6B4235DD                  2
2  038D9FD968DB679259AB                  2
3  03AC693EDD68DA26CEBF                  3
4  040F35F7C6F2D51C7F8E                  4
             customerID          ISIN  user_item_buy_count_1m
0  00523D5E7518C6165B32  GRS003003035                       1
1  00523D5E7518C6165B32  GRS323003012                       1
2  0203622547FA6B4235DD  GRS496003005                       2
3  038D9FD968DB679259AB  GRS495003006                       2
4  03AC693EDD68DA26CEBF  GRS003003035                       1


In [29]:
# Каждый из 500 клиентов × каждый из 50 активов
april_features = pd.DataFrame({
    "customerID": large_client_ids
}).merge(
    pd.DataFrame({"ISIN": large_asset_ids}),
    how="cross"
)

# Присоединяем общую активность клиента
april_features = april_features.merge(
    user_counts,
    on="customerID",
    how="left",
    validate="many_to_one"
)

# Присоединяем покупки конкретного актива
april_features = april_features.merge(
    pair_counts,
    on=["customerID", "ISIN"],
    how="left",
    validate="many_to_one"
)

feature_columns = [
    "user_buy_count_1m",
    "user_item_buy_count_1m"
]

# Нет мартовских покупок — ставим 0
april_features[feature_columns] = (
    april_features[feature_columns].fillna(0).astype(int)
)

april_features["prediction_date"] = pd.Timestamp("2020-04-01")

print(april_features.head().to_string(index=False))
print("Количество строк:", len(april_features))

          customerID         ISIN  user_buy_count_1m  user_item_buy_count_1m prediction_date
485EF2CE5FB9D9A7A08C GRS003003035                  7                       7      2020-04-01
485EF2CE5FB9D9A7A08C GRS015003007                  7                       0      2020-04-01
485EF2CE5FB9D9A7A08C GRS323003012                  7                       0      2020-04-01
485EF2CE5FB9D9A7A08C GRS393503008                  7                       0      2020-04-01
485EF2CE5FB9D9A7A08C GRS014003032                  7                       0      2020-04-01
Количество строк: 25000


In [30]:
# Апрельские покупки выбранных клиентов
april_large_buys = large_buys[
    (large_buys["timestamp"] >= "2020-04-01")
    & (large_buys["timestamp"] < "2020-05-01")
]

# Каждая купленная пара получает ответ 1
april_labels = (
    april_large_buys[["customerID", "ISIN"]]
    .drop_duplicates()
    .copy()
)
april_labels["label"] = 1

# Присоединяем ответы к мартовским признакам
april_dataset = april_features.merge(
    april_labels,
    on=["customerID", "ISIN"],
    how="left",
    validate="one_to_one"
)

# Если апрельской покупки нет — ответ 0
april_dataset["label"] = (
    april_dataset["label"].fillna(0).astype(int)
)

print(april_dataset["label"].value_counts())
print("Количество строк:", len(april_dataset))

label
0    24677
1      323
Name: count, dtype: int64
Количество строк: 25000


In [ ]:
def make_month_dataset(prediction_date):
    cutoff = pd.Timestamp(prediction_date)
    start = cutoff - pd.DateOffset(months=1)
    end = cutoff + pd.DateOffset(months=1)

    # Проверяем дату и наличие месяца для ответа
    if not cutoff.is_month_start or cutoff != cutoff.normalize():
        raise ValueError("Дата должна быть первым числом месяца")

    last_day = transactions["timestamp"].max().normalize()
    if end > last_day + pd.Timedelta(days=1):
        raise ValueError("В данных ещё нет полного месяца для ответа")

    # 1. Прошлый месяц — для признаков
    past = large_buys[
        (large_buys["timestamp"] >= start)
        & (large_buys["timestamp"] < cutoff)
    ]

    # Месяц прогноза — для правильных ответов
    future = large_buys[
        (large_buys["timestamp"] >= cutoff)
        & (large_buys["timestamp"] < end)
    ]

    # 2. Считаем два знакомых признака
    user_counts = past.groupby("customerID").size().reset_index(
        name="user_buy_count_1m"
    )
    pair_counts = past.groupby(["customerID", "ISIN"]).size().reset_index(
        name="user_item_buy_count_1m"
    )

    # 3. Создаём все пары и присоединяем признаки
    result = pd.DataFrame({"customerID": large_client_ids}).merge(
        pd.DataFrame({"ISIN": large_asset_ids}),
        how="cross"
    )
    result = result.merge(
        user_counts, on="customerID",
        how="left", validate="many_to_one"
    )
    result = result.merge(
        pair_counts, on=["customerID", "ISIN"],
        how="left", validate="one_to_one"
    )

    # 4. Добавляем правильные ответы
    labels = future[["customerID", "ISIN"]].drop_duplicates().copy()
    labels["label"] = 1

    result = result.merge(
        labels, on=["customerID", "ISIN"],
        how="left", validate="one_to_one"
    )

    columns = ["user_buy_count_1m", "user_item_buy_count_1m", "label"]
    result[columns] = result[columns].fillna(0).astype(int)
    result["prediction_date"] = cutoff

    return result


# Проверяем функцию на уже знакомом апреле
april_check = make_month_dataset("2020-04-01")

print("Строк:", len(april_check))
print("Пар с покупкой:", april_check["label"].sum())

Строк: 25000
Пар с покупкой: 323


In [ ]:
# Первое число каждого месяца с апреля 2020 по ноябрь 2022
large_prediction_dates = pd.date_range(
    "2020-04-01", "2022-11-01", freq="MS"
)

# Собираем таблицу для каждого месяца
monthly_tables = []

for date in large_prediction_dates:
    monthly_tables.append(make_month_dataset(date))

# Объединяем месяцы в одну таблицу
large_dataset = pd.concat(monthly_tables, ignore_index=True)
del monthly_tables  # Освобождаем временный список

# Обучение: апрель 2020 — декабрь 2021
train = large_dataset[
    large_dataset["prediction_date"] < "2022-01-01"
].copy()

# Выбор модели: январь — июнь 2022
validation = large_dataset[
    (large_dataset["prediction_date"] >= "2022-01-01")
    & (large_dataset["prediction_date"] < "2022-07-01")
].copy()

# Итоговая проверка: июль — ноябрь 2022
test = large_dataset[
    large_dataset["prediction_date"] >= "2022-07-01"
].copy()

print("Всего строк:", len(large_dataset))
print("Обучение:", len(train))
print("Выбор модели:", len(validation))
print("Итоговая проверка:", len(test))

Всего строк: 800000
Обучение: 525000
Выбор модели: 150000
Итоговая проверка: 125000


In [33]:
import numpy as np

# Вероятность покупки по обучающим данным
baseline_probability = train["label"].mean()

# Одинаковый прогноз для всех строк validation
validation_predictions = np.full(
    len(validation), baseline_probability
)

# Метрика качества вероятностного прогноза
def binary_logloss(labels, probabilities):
    y = np.asarray(labels, dtype=float)
    p = np.clip(probabilities, 1e-15, 1 - 1e-15)

    return float(
        -(y * np.log(p) + (1 - y) * np.log(1 - p)).mean()
    )

baseline_score = binary_logloss(
    validation["label"],
    validation_predictions
)

print(f"Вероятность baseline: {baseline_probability:.2%}")
print(f"LogLoss на validation: {baseline_score:.5f}")

Вероятность baseline: 0.82%
LogLoss на validation: 0.04238


In [34]:
from sklearn.ensemble import HistGradientBoostingClassifier

model_features = [
    "user_buy_count_1m",
    "user_item_buy_count_1m"
]

model = HistGradientBoostingClassifier(
    max_iter=100,
    max_leaf_nodes=7,
    early_stopping=False,
    random_state=42
)

# Учимся по признакам и известным ответам
model.fit(train[model_features], train["label"])

# Предсказываем вероятность покупки на validation
model_predictions = model.predict_proba(
    validation[model_features]
)[:, 1]

model_score = binary_logloss(
    validation["label"], model_predictions
)

print(f"Baseline LogLoss: {baseline_score:.5f}")
print(f"Модель LogLoss: {model_score:.5f}")

Baseline LogLoss: 0.04238
Модель LogLoss: 0.03369


In [35]:
# Делим обучающие пары на две группы
bought_before_train = train["user_item_buy_count_1m"] > 0

# Доля будущих покупок внутри каждой группы
repeat_rates = train.groupby(bought_before_train)["label"].mean()

# Для каждой пары validation выбираем вероятность её группы
bought_before_validation = validation["user_item_buy_count_1m"] > 0

repeat_predictions = (
    bought_before_validation
    .map(repeat_rates)
    .fillna(baseline_probability)
    .to_numpy()
)

repeat_score = binary_logloss(
    validation["label"], repeat_predictions
)

comparison = pd.DataFrame({
    "Прогноз": [
        "Одинаковая вероятность",
        "Правило прошлой покупки",
        "Градиентный бустинг"
    ],
    "LogLoss": [
        baseline_score,
        repeat_score,
        model_score
    ]
})

print(comparison.round(5).to_string(index=False))

                Прогноз  LogLoss
 Одинаковая вероятность  0.04238
Правило прошлой покупки  0.03583
    Градиентный бустинг  0.03369


In [36]:
def make_month_dataset_v2(prediction_date):
    # Берём нашу готовую таблицу с двумя признаками
    result = make_month_dataset(prediction_date)

    cutoff = pd.Timestamp(prediction_date)
    start = cutoff - pd.DateOffset(months=1)

    # Покупки всех клиентов за прошлый месяц
    past_all_buys = transactions[
        (transactions["transactionType"] == "Buy")
        & (transactions["timestamp"] >= start)
        & (transactions["timestamp"] < cutoff)
    ]

    # Сколько раз покупали каждый актив
    item_counts = (
        past_all_buys.groupby("ISIN")
        .size()
        .reset_index(name="item_buy_count_1m")
    )

    # Добавляем популярность к каждой паре
    result = result.merge(
        item_counts,
        on="ISIN",
        how="left",
        validate="many_to_one"
    )

    result["item_buy_count_1m"] = (
        result["item_buy_count_1m"].fillna(0).astype(int)
    )

    return result


april_v2 = make_month_dataset_v2("2020-04-01")

print(april_v2[[
    "ISIN",
    "user_item_buy_count_1m",
    "item_buy_count_1m"
]].head().to_string(index=False))

        ISIN  user_item_buy_count_1m  item_buy_count_1m
GRS003003035                       7               2034
GRS015003007                       0               1134
GRS323003012                       0                893
GRS393503008                       0                523
GRS014003032                       0                652


In [37]:
# Пересобираем месяцы для обучения и validation
development_v2 = pd.concat(
    [
        make_month_dataset_v2(date)
        for date in pd.date_range(
            "2020-04-01", "2022-06-01", freq="MS"
        )
    ],
    ignore_index=True
)

train_v2 = development_v2[
    development_v2["prediction_date"] < "2022-01-01"
].copy()

validation_v2 = development_v2[
    development_v2["prediction_date"] >= "2022-01-01"
].copy()

# Теперь используем три признака
features_v2 = [
    "user_buy_count_1m",
    "user_item_buy_count_1m",
    "item_buy_count_1m"
]

# Настройки те же, что у первой модели
model_v2 = HistGradientBoostingClassifier(
    max_iter=100,
    max_leaf_nodes=7,
    early_stopping=False,
    random_state=42
)

model_v2.fit(train_v2[features_v2], train_v2["label"])

predictions_v2 = model_v2.predict_proba(
    validation_v2[features_v2]
)[:, 1]

score_v2 = binary_logloss(
    validation_v2["label"], predictions_v2
)

print(f"Два признака: LogLoss = {model_score:.5f}")
print(f"Три признака: LogLoss = {score_v2:.5f}")

Два признака: LogLoss = 0.03369
Три признака: LogLoss = 0.03153


In [13]:
assets = pd.read_csv(data_dir + "/asset_information.csv")

first_customer = transactions.iloc[0]["customerID"]
first_asset = transactions.iloc[0]["ISIN"]

asset_example = assets[assets["ISIN"] == first_asset]
print("Информация об активе:")
print(asset_example[["ISIN", "assetName", "assetCategory", "timestamp"]].to_string(index=False))

transaction_example = transactions[
    (transactions["customerID"] == first_customer)
    & (transactions["ISIN"] == first_asset)
    & (transactions["timestamp"] >= "2020-03-01")
    & (transactions["timestamp"] < "2020-05-01")
].sort_values("timestamp")

print("\nСделки одного клиента с этим активом:")
print(transaction_example[["timestamp", "transactionType", "totalValue", "units"]].to_string(index=False))


Информация об активе:
        ISIN assetName assetCategory  timestamp
GRS434003000    PPC SA         Stock 2018-01-02

Сделки одного клиента с этим активом:
 timestamp transactionType  totalValue  units
2020-03-27             Buy     11000.0 5000.0
2020-04-06            Sell     12080.0 5000.0
2020-04-24             Buy     13400.0 5000.0


В последней ячейке `==` проверяет равенство для каждой строки таблицы. Выражение в квадратных скобках оставляет подходящие строки. Символ `&` объединяет несколько условий; каждое условие заключено в скобки. `sort_values` сортирует строки по дате. `iloc[0]` выбирает первую строку по позиции.

## Предлагаемая постановка — пока без обучения

На первое число месяца для пары «клиент — доступный актив» оцениваем, купит ли клиент этот актив в начинающемся календарном месяце. Например, прогноз от 1 апреля относится к периоду **[1 апреля, 1 мая)**.

- История: только сделки и информация, доступные **до 1 апреля**.
- Ответ `label=1`: в апреле есть хотя бы одна покупка этого актива клиентом.
- Ответ `label=0`: в апреле в данных нет такой покупки. Это не означает неприязнь, отказ от показа или непригодность инвестиции.

В показанном примере покупка 27 марта — история. Покупка 24 апреля даёт будущий ответ 1. Продажу 6 апреля нельзя использовать как признак прогноза от 1 апреля.

**Выбор на обсуждение:** месячный горизонт позволяет понятно разделить прошлое и будущее. В статье авторы используют другой протокол (шесть месяцев и новые для пользователя активы). Мы допускаем повторные покупки и не будем напрямую сравнивать собственные числа с таблицей статьи.

Следующим шагом определим клиентов и активы-кандидаты по информации, доступной на дату прогноза. Затем построим несколько примеров с ответами 0 и 1. `Buy`/`Sell` из таблицы нельзя просто переименовать в такие ответы.

## Вопрос на понимание

Для прогноза от 1 апреля можем ли мы использовать продажу 6 апреля? Объясни одной фразой.

## Что важно для честного эксперимента

- Авторы добавили часть покупок, реконструируя начальные владения по последующим продажам, и скорректировали некоторые даты. Это обработанный исследовательский набор, а не неизменённый журнал. При подготовке эксперимента нужно решить, как ограничить влияние восстановленных событий; полностью идентифицировать их без флага происхождения может быть невозможно. [Раздел 3.3.1 статьи](https://arxiv.org/html/2407.08692v1#S3.SS3.SSS1).
- `transactionID` встречается у разных клиентов. Это не основание удалять строки по этому полю: единица наблюдения требует отдельного решения. Полностью одинаковых строк в файле нет.
- Метаданные клиентов и активов меняются со временем. Для каждого прогноза выбираем запись, доступную на тот момент, а не последнюю во всей таблице.
- Временной горизонт последнего прогноза должен целиком помещаться в наблюдаемый период.
- Пока построение модели и сравнение признаков не выполнялись.
